# Hypothesis tests and p-values, honestly

Core stage, lesson C8. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** state a null and an alternative hypothesis, read a p-value for what it is and is not, tell a false alarm from a miss and see what power depends on, and see how looking again and again at the same data inflates the false-alarm rate.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.
EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong, tol):
            print("Hint:", message)
    return False


def check_all(label, known):
    """Compare several variables with their known answers: name -> (answer, tolerance). It never stops Run all."""
    got = {name: globals().get(name) for name in known}
    if all(value is None for value in got.values()):
        RESULTS[label] = None
        print(f"{label}: give " + ", ".join(known) + " a value, then run this cell again.")
        return False
    wrong = [name for name, (want, tol) in known.items() if got[name] is None or not same(got[name], want, tol)]
    RESULTS[label] = not wrong
    if wrong:
        print(f"{label}: not yet. Check " + ", ".join(wrong) + ".")
    else:
        print(f"{label}: correct.")
    return not wrong

# One look for every chart of the lesson: small figure, no top and right frame, no legend frame, space made automatically.
BLUE, GREEN, PURPLE, GOLD, RED = "#0072b2", "#00a07a", "#9a6be0", "#ad9100", "#c0392b"  # the colors of the site's figures
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[BLUE, GREEN, PURPLE, GOLD]),
    "figure.figsize": (4.5, 2.8),
    "figure.constrained_layout.use": True,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
users = pd.read_csv(BASE + "users.csv")
assign = pd.read_csv(BASE + "experiment_assignments.csv", parse_dates=["assigned_at"])
events = pd.read_csv(BASE + "events.csv", usecols=["user_id", "event_name", "platform"])    # the biggest file, 13 MB: a few seconds

# Known answer from the dataset README: 21,841 people and 15,868 experiment rows.
assert (len(users), len(assign)) == (21841, 15868)

# The experiment of this lesson: welcome_tips, tips on the first screens. A person is in at most one experiment.
tips = assign[assign["experiment"] == "welcome_tips"].merge(users, on="user_id")
# The platform of a person is the platform of their signup_complete event.
signup_platform = events[events["event_name"] == "signup_complete"].drop_duplicates("user_id").set_index("user_id")["platform"]
tips["platform"] = tips["user_id"].map(signup_platform)
tips["day"] = (tips["assigned_at"].dt.normalize() - pd.Timestamp("2026-03-16")).dt.days + 1      # day 1 is March 16
is_var = (tips["variant"] == "variant").to_numpy()              # True for the people who saw the tips
y = tips["converted"].to_numpy().astype(int)                    # 1 = bought Plus within 14 days of signing up
assert (len(tips), int(is_var.sum()), int(y.sum())) == (4968, 2519, 610)


def two_prop_p(c0, n0, c1, n1):
    """Two-sided p-value for 'the two groups share one rate': c = buyers, n = people, group 1 against group 0.
    The next lesson derives it; here it is a tool. It works on single numbers and on whole arrays of experiments."""
    pooled = (c0 + c1) / (n0 + n1)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n0 + 1 / n1))
    z = (c1 / n1 - c0 / n0) / se
    return 2 * stats.norm.sf(np.abs(z))


print(f"Data loaded: {len(tips):,} people in welcome_tips, {int(is_var.sum()):,} saw the tips, {int(y.sum())} bought Plus within 14 days.")

## Unit 1: Say what nothing going on means, and shuffle it

**Predict first.** The cell shuffles the variant labels 10,000 times, so that who saw the tips is arbitrary, and records the gap in conversion each time. Will the real gap sit near the middle of the shuffled gaps or far out in a tail? About how many of the 10,000 shuffles will give a gap at least as far from 0?

In [ ]:
n0, n1 = int((~is_var).sum()), int(is_var.sum())                  # people in control and in the variant
c0, c1 = int(y[~is_var].sum()), int(y[is_var].sum())              # buyers in control and in the variant
p0, p1 = c0 / n0, c1 / n1                                         # conversion rates
gap = p1 - p0                                                     # variant minus control
print(n0, n1, c0, c1, round(p0, 4), round(p1, 4), round(float(gap), 4))
rng = np.random.default_rng(42)
shuffled = np.empty(10_000)
for k in range(10_000):
    mix = rng.permutation(is_var)                                 # shuffle who is "variant": now the label means nothing
    shuffled[k] = y[mix].mean() - y[~mix].mean()                  # the gap this shuffle produces
extreme = int((np.abs(shuffled) >= abs(gap)).sum())               # shuffles at least as far from 0 as the real gap, either way
print(extreme, round((extreme + 1) / (len(shuffled) + 1), 3))                    # the p-value: the share of shuffles that are this extreme

## Unit 2: What a p-value says, and what it does not

**Predict first.** The p-value of unit 1 is a share of shuffles. The cell turns the same gap into a z-score and reads the p-value from the bell curve, then builds the 95 percent interval of the difference, in percentage points. Will the interval contain 0? Will it rule out a lift of +1 point?

In [ ]:
se_null = shuffled.std(ddof=1)                                   # the spread of the shuffled gaps: the standard error if nothing is going on
z = gap / se_null                                                # how many standard errors from 0 the real gap lies
p_norm = 2 * stats.norm.sf(abs(z))                               # both tails of the bell curve
print(round(float(se_null), 4), round(float(z), 3), round(float(p_norm), 3))
print(round(float(two_prop_p(c0, n0, c1, n1)), 3))               # the formula of the next lesson, the same idea
se_diff = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)       # the standard error of the difference, from the two groups
low, high = gap - 1.96 * se_diff, gap + 1.96 * se_diff           # the 95 percent interval of the difference
print(round(float(100 * gap), 2), round(float(100 * low), 2), round(float(100 * high), 2))      # in percentage points

## Unit 3: Two ways to be wrong, and power

**Predict first.** The cell simulates 2,000 experiments for each group size. In the no-effect experiments, what share will have p below 0.05, whatever the size? And in the experiments with a real lift of 2 points, how does the share caught change when each group grows from 1,250 to 10,000 people, and when the bar goes from 0.05 to 0.01?

In [ ]:
rng = np.random.default_rng(1)
p_base = 0.124                                                   # a baseline conversion rate: control's 12.4 percent


def experiments(size, p_variant, reps=2000):
    """p-values of `reps` simulated experiments with `size` people in each group."""
    buyers0 = rng.binomial(size, p_base, size=reps)
    buyers1 = rng.binomial(size, p_variant, size=reps)
    return two_prop_p(buyers0, size, buyers1, size)


table = {}
for size in (1250, 2500, 5000, 10000):
    none = experiments(size, 0.124)                              # no effect: the null is true
    lift = experiments(size, 0.144)                              # a real lift of 2 points, 12.4 to 14.4 percent
    table[size] = (float((none < 0.05).mean()), [float((lift < a).mean()) for a in (0.05, 0.01)])
    if size == 2500:
        first_100 = none[:100]                                   # kept for the next cell
    print(size, round(table[size][0], 3), [round(share, 3) for share in table[size][1]])

Now the experiments of size 2,500 in detail, and what "significant" is worth when real lifts are rare.

In [ ]:
print(int((first_100 < 0.05).sum()), "of the first 100 no-effect experiments of size 2,500 have p < 0.05")
power = table[2500][1][0]                                        # the share of real lifts caught at size 2,500, from the cell above
real, false = 100 * power, 900 * 0.05                            # 1,000 tests: 100 with a real lift, 900 with none
print(round(real, 1), round(false, 1), round(real / (real + false), 2))

## Unit 4: Looking again and again

**Predict first.** The cell looks at welcome_tips after each day from day 7 to day 42 (36 looks) and records the p-value each time. Will it ever drop below 0.05? On which day first, and on how many of the 36 looks?

In [ ]:
rows = []
for day in range(7, 43):                                         # a look after each day from day 7 to day 42
    d = tips[tips["day"] <= day]
    v = d["variant"] == "variant"
    rows.append((day, len(d), float(two_prop_p(d.loc[~v, "converted"].sum(), (~v).sum(), d.loc[v, "converted"].sum(), v.sum()))))
running = pd.DataFrame(rows, columns=["day", "people", "p"])
below = running[running["p"] < 0.05]
print(len(running), len(below), int(below["day"].iloc[0]), round(float(below["p"].iloc[0]), 3), round(float(running["p"].iloc[-1]), 3))

**Predict first.** The cell cuts the same experiment into 12 segments (5 channels, 3 platforms, 4 regions) and tests each one. How many of the 12 will have p below 0.05? Will any survive the stricter bar of 0.05 divided by 12?

In [ ]:
segments = []
for column in ["channel", "platform", "region"]:
    for value, d in tips.groupby(column):
        v = d["variant"] == "variant"
        p = float(two_prop_p(d.loc[~v, "converted"].sum(), (~v).sum(), d.loc[v, "converted"].sum(), v.sum()))
        segments.append((column, value, len(d), round(float(d.loc[~v, "converted"].mean()), 3), round(float(d.loc[v, "converted"].mean()), 3), p))
seg = pd.DataFrame(segments, columns=["split", "segment", "people", "control", "variant", "p"])
print(len(seg), int((seg["p"] < 0.05).sum()))
print(seg[seg["p"] < 0.05].round(3).to_string(index=False))
bar = 0.05 / len(seg)                                            # the stricter bar when 12 looks are made
print(round(float(bar), 4), int((seg["p"] < bar).sum()), round(float(1 - 0.95 ** len(seg)), 3))

## Finish it

### Exercise 1: Finish it

**Task:** Did the welcome tips change the share of people who finished onboarding (the `onboarding_complete` event)? Count the shuffles at least as far from 0 as the real gap, in either direction, and turn the count into a p-value: (count + 1) / (10,000 + 1), rounded to 3 decimals.

**Expected:** `answer_1` is a number between 0 and 1.

In [ ]:
finished = tips["user_id"].isin(events.loc[events["event_name"] == "onboarding_complete", "user_id"]).to_numpy().astype(int)    # given: 1 = finished the first-run screens
gap_f = finished[is_var].mean() - finished[~is_var].mean()                                                                   # given: the real gap, variant minus control
rng = np.random.default_rng(3)
shuffled_f = np.array([finished[m].mean() - finished[~m].mean() for m in (rng.permutation(is_var) for _ in range(10_000))])   # given: 10,000 shuffles
# the p-value: (the shuffles at least as far from 0 as gap_f, plus 1) / (10,000 + 1), rounded to 3 decimals
answer_1 = None  # replace None with the p-value

In [ ]:
_ = check("Exercise 1", "answer_1", 0.694, traps={0.341: "That counts only the shuffles with a gap at least as large in the positive direction. The p-value is two-sided: count the gaps that are as far from 0 in either direction (np.abs)."}, tol=0.03)

<details>
<summary>Hint</summary>

Unit 1 counted `(np.abs(shuffled) >= abs(gap)).sum()`. Use the same count with `shuffled_f` and `gap_f`.

</details>

<details>
<summary>Solution</summary>

```python
extreme_f = int((np.abs(shuffled_f) >= abs(gap_f)).sum())
answer_1 = round((extreme_f + 1) / (len(shuffled_f) + 1), 3)
```

70.1 percent of the control group and 70.6 percent of the variant finished the first-run screens, a gap of 0.55 points; the p-value is 0.694, so there is no evidence that the tips changed it. The check cell warns about counting one tail only, which gives 0.341.

</details>

## On your own

### Exercise 2: On your own

**Task:** Welcome tips would be worth keeping if they lifted conversion from 12.4 to 13.4 percent, 1 point. How many people in each group would a test need to catch that lift at least 80 percent of the time with a bar of 0.05? Before you start, say in one sentence what the answer looks like: one group size. Use `two_prop_p` and the simulation of unit 3, with 2,000 experiments for each of the sizes 5,000, 10,000, 20,000 and 40,000, and set `answer_2` to the smallest size whose share of experiments with p below 0.05 is at least 0.80. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is one of the four sizes.

In [ ]:
_ = check("Exercise 2", "answer_2", 20000, traps={10000: "At 10,000 people in each group the test catches a 1 point lift only about half of the time. Print the share for every size, then pick the smallest that reaches 0.80."})

<details>
<summary>Hint</summary>

Copy the cell of unit 3: `rng.binomial(size, 0.124, size=2000)` for the buyers of one group and `rng.binomial(size, 0.134, size=2000)` for the other, then `two_prop_p`, then the share of p-values below 0.05. Loop over the four sizes.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"A share for each of the four sizes, rising with the size; one of them is the first to reach 0.80."*

```python
for size in (5000, 10000, 20000, 40000):
    buyers0 = rng.binomial(size, 0.124, size=2000)
    ...
```

</details>

<details>
<summary>Solution</summary>

```python
def power_at(size, p_variant, p_control=0.124, reps=2000):
    rng = np.random.default_rng(5)
    buyers0 = rng.binomial(size, p_control, size=reps)
    buyers1 = rng.binomial(size, p_variant, size=reps)
    return float((two_prop_p(buyers0, size, buyers1, size) < 0.05).mean())


powers = {size: power_at(size, 0.134) for size in (5000, 10000, 20000, 40000)}
answer_2 = min(size for size, power in powers.items() if power >= 0.80)
```

At 5,000 people in each group the test catches a 1 point lift in 32 percent of the experiments, at 10,000 in 54 percent, at 20,000 in 84 percent and at 40,000 in 99 percent: the smallest size that reaches 80 percent is 20,000 in each group, about eight times as many people as welcome_tips had.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague wants to know how often looking every day and stopping at the first p-value below 0.05 gives a false alarm. They simulate 500 experiments with no effect at all, each with 36 daily looks from day 7 to day 42, and report the share of looks with p below 0.05. That share answers another question. Fix the last line so that `answer_3` is the share of the 500 experiments in which at least one of the 36 looks has p below 0.05, rounded to 3 decimals.

**Expected:** `answer_3` is a number close to what the check cell expects: it accepts a difference of 0.05, because the number depends on the seed.

In [ ]:
rng = np.random.default_rng(11)
per_group = 59                                                   # given: people a day in each group, 118 a day in all
days = np.arange(1, 43)                                          # given: 42 days, no effect at all: both groups buy at 12.4 percent
c0 = np.cumsum(rng.binomial(per_group, 0.124, size=(500, 42)), axis=1)      # given: buyers in control after each day, 500 experiments
c1 = np.cumsum(rng.binomial(per_group, 0.124, size=(500, 42)), axis=1)      # given: buyers in the variant after each day
p_run = two_prop_p(c0, per_group * days, c1, per_group * days)             # given: 500 rows of 42 running p-values
looks = p_run[:, 6:]                                             # given: the 36 looks from day 7 on
# colleague: "the false-alarm rate of daily peeking is the share of looks with p < 0.05"
answer_3 = round(float((looks < 0.05).mean()), 3)

In [ ]:
_ = check("Exercise 3", "answer_3", 0.238, traps={0.061: "That is the share of looks, not of experiments. An experiment gives a false alarm if any of its looks is below 0.05: use .any(axis=1) on the 500 x 36 table."}, tol=0.05)

<details>
<summary>Hint</summary>

`looks < 0.05` is a table of 500 rows (experiments) and 36 columns (looks). The colleague's `.mean()` averages all 18,000 cells. Which axis gives one yes or no for each experiment?

</details>

<details>
<summary>Solution</summary>

```python
rng = np.random.default_rng(11)
per_group = 59
days = np.arange(1, 43)
c0 = np.cumsum(rng.binomial(per_group, 0.124, size=(500, 42)), axis=1)
c1 = np.cumsum(rng.binomial(per_group, 0.124, size=(500, 42)), axis=1)
p_run = two_prop_p(c0, per_group * days, c1, per_group * days)
looks = p_run[:, 6:]
answer_3 = round(float((looks < 0.05).any(axis=1).mean()), 3)
```

Of the 18,000 looks, 6.1 percent are below 0.05, which is what the colleague reported. But 119 of the 500 experiments (23.8 percent) have at least one look below 0.05, and that is the false-alarm rate of someone who stops at the first one. The check cell reports the colleague's 0.061 as a hint.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")